GET Request to pull Federal Register Documents

In [19]:
import requests
import pandas as pd

In [20]:
url = "https://www.federalregister.gov/api/v1/documents.json"

params = {
    "conditions[term]": '"artificial intelligence"', 
    "conditions[type][]": ["RULE"],
    "conditions[publication_date][gte]": "2017-01-20",
    "per_page": 1000,
    "order": "oldest",
    "fields[]": ["document_number", "title", "type", "abstract",
                 "publication_date", "agencies", "html_url", "raw_text_url"],
}

In [21]:
# Download all rules (we have 183)
response = requests.get(url, params=params)
response.raise_for_status()           # stop with an error if the request failed
data = response.json()

all_results = data["results"]
print("Downloaded:", len(all_results), "| API reports:", data["count"])

# Check: did we get every document? (Fails if results ever exceed one page)
assert len(all_results) == data["count"]

Downloaded: 183 | API reports: 183


In [22]:
# Turn the results into a table
df = pd.DataFrame(all_results)

# "agencies" holds a list per document; keep just the first agency's name
agency_names = []
for agencies in df["agencies"]:
    if agencies:                              # the list is not empty
        agency_names.append(agencies[0]["name"])
    else:
        agency_names.append(None)
df["agency"] = agency_names

In [23]:
# Label each document's administration
# Dates look like "2023-10-30". Text dates in this format compare correctly, so "2020-05-01" < "2021-01-20" is True.

def label_admin(date):
    if date < "2021-01-20":
        return "Trump I"      # Jan 20, 2017 – Jan 19, 2021
    elif date < "2025-01-20":
        return "Biden"        # Jan 20, 2021 – Jan 19, 2025
    else:
        return "Trump II"     # Jan 20, 2025 – today

df["administration"] = df["publication_date"].apply(label_admin)

# Quick look: how many documents per administration?
print(df["administration"].value_counts())

administration
Biden       101
Trump II     62
Trump I      20
Name: count, dtype: int64


In [24]:
# Check and save
assert df["document_number"].is_unique    # no document appears twice

# Drop the list column (CSV can't store lists well); we kept "agency" instead
df = df.drop(columns=["agencies"])

# Save so other notebooks can load it
df.to_csv("fr_ai_rules.csv", index=False)
print("Saved", len(df), "documents")

Saved 183 documents


In [25]:
# Earliest and latest publication dates in our data
print("First document:", df["publication_date"].min())
print("Last document:", df["publication_date"].max())

First document: 2018-11-23
Last document: 2026-10-13


In [26]:
# Sort by date (oldest first) and show the first 5 documents
df.sort_values("publication_date")[["publication_date", "type", "title"]].head(5)

,publication_date,type,title
0,2018-11-23,Rule,Medicare Program; Revisions to Payment Policie...
1,2019-07-08,Rule,Designation of Music Licensing Collective and ...
2,2019-08-16,Rule,Medicare Program; Hospital Inpatient Prospecti...
3,2019-09-13,Rule,Policy on No-Action Letters
4,2019-11-12,Rule,Medicare Program: Changes to Hospital Outpatie...


---
# Part 2: Label AI relevance with Gemini (Week 7 lab)

Every rule above contains the phrase "artificial intelligence", but many mention it only once (for example, a Medicare payment rule). Here Gemini labels each rule as **central / passing / irrelevant** to AI, using the same codebook we use by hand.

**Steps**
1. Pick 10 rules at random and pull the passages where "artificial intelligence" appears, which gives `lab07_team_texts.csv` (`id`, `text`).
2. Two of us label the same 10 rules on our own in `lab07_hand_labels.csv`.
3. Gemini labels the same 10.
4. Compare, and read every disagreement.

**Codebook**

| Label | Definition |
|---|---|
| `central` | AI is a main subject of the rule: it regulates, defines, permits, restricts, funds, or sets policy for AI, or spends a substantial part of its text on AI. |
| `passing` | AI is discussed as a technology, but the rule is mainly about something else (e.g., AI is one item in a list of technologies, or the rule briefly answers a public comment about AI). |
| `irrelevant` | The phrase appears without any discussion of AI as a technology (e.g., only in the title of a cited report, an office's name, or a URL). |
| `unclear` | The text given is not enough to decide. |

Setup (one time, in a terminal): `pip install google-genai python-dotenv beautifulsoup4`, and put `GEMINI_API_KEY=your_key_here` in a `.env` file next to this notebook. Keep `.env` out of Git.

In [27]:
import json
import os
import re
import time

from bs4 import BeautifulSoup

# Reload the saved table so Part 2 can run without re-running the GET request
df = pd.read_csv("fr_ai_rules.csv")

# Pick 10 rules at random. A fixed random_state gives everyone the same 10.
sample = df.sample(n=10, random_state=211).reset_index(drop=True)
sample[["document_number", "publication_date", "administration", "agency", "title"]]

,document_number,publication_date,administration,agency,title
0,2023-03986,2023-03-15,Biden,Energy Department,Energy Conservation Program: Test Procedure fo...
1,2026-16981,2026-08-20,Trump II,Justice Department,Application for Relief From Disabilities Impos...
2,2024-27479,2024-12-13,Biden,Federal Communications Commission,Access to Video Conferencing
3,2025-01422,2025-01-21,Trump II,Homeland Security Department,Ratification of Security Directives
4,2024-22587,2024-10-02,Biden,Commerce Department,Expansion of Validated End User Authorization:...
5,2023-24293,2023-11-22,Biden,Health and Human Services Department,Medicare Program: Hospital Outpatient Prospect...
6,2026-07663,2026-04-20,Trump II,Justice Department,Extension of Compliance Dates for Nondiscrimin...
7,2024-06810,2024-04-12,Biden,Personnel Management Office,Pathways Programs
8,2022-27675,2022-12-29,Biden,Securities and Exchange Commission,Insider Trading Arrangements and Related Discl...
9,2026-05676,2026-03-24,Trump II,Health and Human Services Department,Administrative Simplification; Adoption of Sta...


### Pull the passages that mention AI

Full rules can run hundreds of pages, which is too long to read by hand or send to Gemini. For each rule we download the full text, then keep only the sentences that say "artificial intelligence", plus one sentence before and after for context. We also count the mentions and the document's length, since both help separate `central` from `passing`.

In [28]:
AI_PATTERN = re.compile(r"artificial intelligence", re.IGNORECASE)

def get_full_text(raw_text_url):
    """Download a rule's full text and collapse line breaks into single spaces."""
    r = requests.get(raw_text_url, timeout=30)
    r.raise_for_status()
    text = BeautifulSoup(r.text, "html.parser").get_text()   # strips HTML tags if there are any
    return " ".join(text.split())

def ai_passages(text, max_passages=6, max_chars=1200):
    """Return passages around each sentence that mentions artificial intelligence."""
    # Split into sentences at . ! or ? followed by a space and a capital letter (rough, but good enough)
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z(\"])", text)

    # Positions of the sentences that mention AI, plus the sentence before and after each one
    keep = set()
    for i, sentence in enumerate(sentences):
        if AI_PATTERN.search(sentence):
            keep.update([i - 1, i, i + 1])
    keep = sorted(i for i in keep if 0 <= i < len(sentences))

    # Join neighbouring sentences into one passage so overlapping windows aren't repeated
    passages = []
    current = []
    for i in keep:
        if current and i != current[-1] + 1:
            passages.append(" ".join(sentences[j] for j in current))
            current = []
        current.append(i)
    if current:
        passages.append(" ".join(sentences[j] for j in current))

    # Keep the prompt short: at most 6 passages, each cut to 1,200 characters
    return [p[:max_chars] for p in passages[:max_passages]]

def build_text(row, full_text):
    """Combine the metadata and AI passages into the one text we label."""
    abstract = row["abstract"] if isinstance(row["abstract"], str) else "(none)"
    passages = ai_passages(full_text)
    lines = [
        f"Title: {row['title']}",
        f"Agency: {row['agency']}",
        f"Abstract: {abstract}",
        f"Length: {len(full_text.split()):,} words",
        f"Times 'artificial intelligence' appears: {len(AI_PATTERN.findall(full_text))}",
        "",
        "Passages that mention artificial intelligence:",
    ]
    for k, passage in enumerate(passages, start=1):
        lines.append(f"[{k}] {passage}")
    return "\n".join(lines)

In [29]:
texts = []
n_mentions = []
for _, row in sample.iterrows():
    full_text = get_full_text(row["raw_text_url"])
    texts.append(build_text(row, full_text))
    n_mentions.append(len(AI_PATTERN.findall(full_text)))
    time.sleep(0.5)                       # be polite to the Federal Register server

sample["text"] = texts
sample["n_mentions"] = n_mentions

# The lab file: two columns, id and text
team_texts = sample[["document_number", "text"]].rename(columns={"document_number": "id"})
team_texts.to_csv("lab07_team_texts.csv", index=False)
print("Saved", len(team_texts), "texts")

# Read one before sending anything: check there are no names of private people
print(team_texts.loc[0, "text"])

Saved 10 texts
Title: Energy Conservation Program: Test Procedure for Television Sets
Agency: Energy Department
Abstract: This final rule amends the test procedure for television sets to incorporate by reference the relevant updated industry standard. The Department of Energy ("DOE") has determined that incorporating the updated industry standard will result in a test procedure that is more representative of the average energy use of television sets.
Length: 31,124 words
Times 'artificial intelligence' appears: 1

Passages that mention artificial intelligence:
[1] In the March 2022 NOPR, DOE also noted that it was aware of certain preset picture settings being introduced on recent TVs that are known to adapt the TV's configuration based on the content, usage pattern, and the environment in which the TV operates. These TVs use artificial intelligence technology to adapt and adjust these settings and such a preset picture setting is sometimes available in addition to Filmmaker mode (defi

### Hand-label first, before looking at Gemini's labels

The next cell makes `lab07_hand_labels.csv` with two empty columns. Two of us each fill in our own column on our own (in Excel or Google Sheets), using the codebook above and the text in `lab07_team_texts.csv` (or the `html_url`). The cell won't overwrite the file once it exists, so re-running it is safe.

In [30]:
if not os.path.exists("lab07_hand_labels.csv"):
    hand = sample[["document_number", "title", "html_url", "n_mentions"]].rename(columns={"document_number": "id"})
    hand["label_coder1"] = ""
    hand["label_coder2"] = ""
    hand.to_csv("lab07_hand_labels.csv", index=False)
    print("Created lab07_hand_labels.csv; fill in label_coder1 and label_coder2")
else:
    print("lab07_hand_labels.csv already exists; not overwriting")

lab07_hand_labels.csv already exists; not overwriting


### Gemini setup

In [31]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()               # load GEMINI_API_KEY from .env
GEMINI_MODEL = "gemini-3.5-flash-lite"

# If Gemini returns 429 (too many requests) or 503, wait and try again, up to 3 tries
RETRY = types.HttpRetryOptions(
    attempts=3,
    initial_delay=1.0,
    exp_base=2,
    http_status_codes=[429, 503],
)

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"],
    http_options=types.HttpOptions(retry_options=RETRY),
)

In [32]:
# System instruction: the task and codebook (the same rules we use by hand)
SYSTEM = """You label U.S. Federal Register rules by how relevant they are to artificial intelligence (AI).
Every rule contains the phrase "artificial intelligence" at least once. You will see its title, agency,
abstract, length, how many times the phrase appears, and the passages where it appears.

Give the rule one label:
central: AI is a main subject of the rule. The rule regulates, defines, permits, restricts, funds, or sets policy for AI, or spends a substantial part of its text on AI.
passing: AI is discussed as a technology, but the rule is mainly about something else. For example, AI is one item in a list of technologies, or the rule briefly answers a public comment about AI.
irrelevant: The phrase appears without any discussion of AI as a technology, for example only in the title of a cited report, the name of an office, or a URL.
unclear: The text given is not enough to decide.

Judge only from the text given.
Also give a reason of one short sentence.
"""

# Schema: Gemini must return JSON with one of our labels and a reason
LABELS = ["central", "passing", "irrelevant", "unclear"]
LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": LABELS},
        "reason": {"type": "string"},
    },
    "required": ["label", "reason"],
}

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema=LABEL_SCHEMA,
    temperature=0,
)

In [33]:
# Try one rule first and read the answer before running all 10
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=team_texts.loc[0, "text"],
    config=CONFIG,
)
print(response.candidates[0].finish_reason)   # should be STOP
print(json.loads(response.text))

FinishReason.STOP
{'label': 'passing', 'reason': 'Artificial intelligence is only briefly mentioned regarding adaptive TV picture settings in a test procedure rule for televisions.'}


In [34]:
# Label all 10 rules, one API call per rule
team_texts = pd.read_csv("lab07_team_texts.csv")

results = []
for _, row in team_texts.iterrows():
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=row["text"],
        config=CONFIG,
    )
    try:
        result = json.loads(response.text)
        label, reason = result["label"], result["reason"]
    except (TypeError, ValueError, KeyError):
        # The reply was cut off or blocked; record it so we can check it by hand
        label = "error"
        reason = str(response.candidates[0].finish_reason)
    results.append({"id": row["id"], "model_label": label, "model_reason": reason})

gemini = pd.DataFrame(results)
gemini.to_csv("lab07_gemini_labels.csv", index=False)
gemini

,id,model_label,model_reason
0,2023-03986,passing,Artificial intelligence is mentioned only brie...
1,2026-16981,passing,Artificial intelligence is only mentioned in a...
2,2024-27479,passing,The mention of artificial intelligence is just...
3,2025-01422,unclear,The provided excerpt cuts off before mentionin...
4,2024-22587,passing,Artificial intelligence is mentioned only as a...
5,2023-24293,passing,Artificial intelligence is briefly discussed i...
6,2026-07663,passing,Artificial intelligence is only mentioned in t...
7,2024-06810,passing,Artificial intelligence is mentioned only once...
8,2022-27675,passing,Artificial intelligence is only briefly mentio...
9,2026-05676,unclear,The provided text does not contain any passage...


### Compare our labels with Gemini's

Run this after both coders have filled in `lab07_hand_labels.csv`. For each disagreement, decide: did one of us misread the text, did Gemini get it wrong, or is the definition unclear? If it's the definition, edit the codebook (and `SYSTEM`) and label again.

In [35]:
hand = pd.read_csv("lab07_hand_labels.csv")
gemini = pd.read_csv("lab07_gemini_labels.csv")
compare = hand.merge(gemini, on="id")

# How often do we agree?
print("Coder 1 vs coder 2:", (compare["label_coder1"] == compare["label_coder2"]).mean())
print("Coder 1 vs Gemini: ", (compare["label_coder1"] == compare["model_label"]).mean())
print("Coder 2 vs Gemini: ", (compare["label_coder2"] == compare["model_label"]).mean())

print(pd.crosstab(compare["label_coder1"], compare["model_label"]))

# Every row where anyone disagrees
disagree = compare[(compare["label_coder1"] != compare["label_coder2"]) |
                   (compare["label_coder1"] != compare["model_label"])]
disagree[["id", "title", "n_mentions", "label_coder1", "label_coder2", "model_label", "model_reason"]]

Coder 1 vs coder 2: 0.0
Coder 1 vs Gemini:  0.0
Coder 2 vs Gemini:  0.0
Empty DataFrame
Columns: []
Index: []


,id,title,n_mentions,label_coder1,label_coder2,model_label,model_reason
0,2023-03986,Energy Conservation Program: Test Procedure fo...,1,NaN,NaN,passing,Artificial intelligence is mentioned only brie...
1,2026-16981,Application for Relief From Disabilities Impos...,1,NaN,NaN,passing,Artificial intelligence is only mentioned in a...
2,2024-27479,Access to Video Conferencing,1,NaN,NaN,passing,The mention of artificial intelligence is just...
3,2025-01422,Ratification of Security Directives,1,NaN,NaN,unclear,The provided excerpt cuts off before mentionin...
4,2024-22587,Expansion of Validated End User Authorization:...,1,NaN,NaN,passing,Artificial intelligence is mentioned only as a...
5,2023-24293,Medicare Program: Hospital Outpatient Prospect...,3,NaN,NaN,passing,Artificial intelligence is briefly discussed i...
6,2026-07663,Extension of Compliance Dates for Nondiscrimin...,2,NaN,NaN,passing,Artificial intelligence is only mentioned in t...
7,2024-06810,Pathways Programs,1,NaN,NaN,passing,Artificial intelligence is mentioned only once...
8,2022-27675,Insider Trading Arrangements and Related Discl...,1,NaN,NaN,passing,Artificial intelligence is only briefly mentio...
9,2026-05676,Administrative Simplification; Adoption of Sta...,0,NaN,NaN,unclear,The provided text does not contain any passage...
